# 🛠️ Notebook 2: Chess — Pieces, Board, Simple Moves

We'll implement King, Rook, Knight, Pawn to keep the code short, plus a mini game loop.
(Queen/Bishop are left as exercises.)

## 🛠️ Setup

```bash
cd 07-object-oriented-design/chess
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass

WHITE, BLACK = 'W', 'B'

class Piece(ABC):
    def __init__(self, color):
        self.color = color
    @abstractmethod
    def symbol(self): ...
    @abstractmethod
    def valid_moves(self, board, r, c): ...
    def __repr__(self):
        s = self.symbol()
        return s.upper() if self.color == WHITE else s.lower()

def on_board(r, c): return 0 <= r < 8 and 0 <= c < 8

class King(Piece):
    def symbol(self): return 'k'
    def valid_moves(self, board, r, c):
        out = []
        for dr in (-1,0,1):
            for dc in (-1,0,1):
                if dr==0 and dc==0: continue
                nr, nc = r+dr, c+dc
                if on_board(nr,nc) and (board[nr][nc] is None or board[nr][nc].color != self.color):
                    out.append((nr,nc))
        return out

class Rook(Piece):
    def symbol(self): return 'r'
    def valid_moves(self, board, r, c):
        out = []
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            while on_board(nr,nc):
                if board[nr][nc] is None:
                    out.append((nr,nc))
                else:
                    if board[nr][nc].color != self.color: out.append((nr,nc))
                    break
                nr, nc = nr+dr, nc+dc
        return out

class Knight(Piece):
    def symbol(self): return 'n'
    def valid_moves(self, board, r, c):
        deltas = [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]
        out = []
        for dr,dc in deltas:
            nr, nc = r+dr, c+dc
            if on_board(nr,nc) and (board[nr][nc] is None or board[nr][nc].color != self.color):
                out.append((nr,nc))
        return out

class Pawn(Piece):
    def symbol(self): return 'p'
    def valid_moves(self, board, r, c):
        dr = -1 if self.color == WHITE else 1  # white moves up (toward row 0)
        out = []
        # forward 1
        if on_board(r+dr, c) and board[r+dr][c] is None:
            out.append((r+dr, c))
        # captures diagonally
        for dc in (-1, 1):
            nr, nc = r+dr, c+dc
            if on_board(nr,nc) and board[nr][nc] and board[nr][nc].color != self.color:
                out.append((nr,nc))
        return out


## Board + printing

In [ ]:
def empty_board():
    return [[None]*8 for _ in range(8)]

def show(board):
    for r, row in enumerate(board):
        print(' '.join(str(p) if p else '.' for p in row))
    print()

b = empty_board()
b[0][4] = King(BLACK);  b[7][4] = King(WHITE)
b[0][0] = Rook(BLACK);  b[7][0] = Rook(WHITE)
b[0][1] = Knight(BLACK);b[7][1] = Knight(WHITE)
for c in range(8): b[1][c] = Pawn(BLACK); b[6][c] = Pawn(WHITE)
show(b)


## Ask pieces for legal moves

In [ ]:
print('White pawn at (6,3) can move to:', b[6][3].valid_moves(b, 6, 3))
print('White rook at (7,0) can move to:', b[7][0].valid_moves(b, 7, 0))
print('Black knight at (0,1) can move to:', b[0][1].valid_moves(b, 0, 1))


## A 2-ply game loop (illustrative)

In [ ]:
class Game:
    def __init__(self, board):
        self.board = board
        self.turn = WHITE
    def move(self, fr, to):
        r,c = fr; nr,nc = to
        p = self.board[r][c]
        if p is None or p.color != self.turn:
            raise ValueError('not your piece')
        if (nr,nc) not in p.valid_moves(self.board, r, c):
            raise ValueError('illegal move')
        self.board[nr][nc] = p
        self.board[r][c] = None
        self.turn = BLACK if self.turn == WHITE else WHITE

g = Game(b)
g.move((6,3), (5,3))   # white pawn forward
g.move((1,3), (2,3))   # black pawn forward
show(g.board)
try:
    g.move((7,1), (6,0))  # not a knight-shape move
except ValueError as e:
    print('expected:', e)


### What we deliberately skipped
- Check / checkmate / stalemate detection.
- Castling, en-passant, promotion.
- Turn timers, draw by repetition.

The point here is the OOP shape — polymorphic `valid_moves`, Board as dumb 8x8 store, Game as state machine.